In [ ]:
from skimage import io
from scipy.ndimage import distance_transform_edt
from skimage.measure import regionprops, label
from skimage.morphology import remove_small_objects
import os
import PIL
import pandas as pd

import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
PIL.Image.MAX_IMAGE_PIXELS = None

In [ ]:
rdir = '/srv/data/michielvc/data/other_projects/christian/final_data/segments/'
vein_dir = os.path.join(rdir, 'veins')
glul_dir = os.path.join(rdir, 'glul')
tissue_dir = os.path.join(rdir, 'tissue')
vsig4_dir = os.path.join(rdir, 'kcs/vsig4/')
f480_dir = os.path.join(rdir, 'kcs/f480/')
crap_dir = os.path.join(rdir, 'crap/')
f480_subset_dir = os.path.join(rdir, 'kcs/f480_subset/')
files = os.listdir(tissue_dir)

data_dict_vsig4 = {
    'id':[],
    'scene':[],
    'h0_avg':[],
    'h0_median':[],
    'h1_avg':[],
    'h1_median':[],
    'tissue_area':[],
    'hole_area':[],
    'kc_area':[],
    'kc_nb': [],
    'avg_kc_area':[],
    'median_kc_area':[]
}

data_dict_f480 = {
    'id':[],
    'scene':[],
    'h0_avg':[],
    'h0_median':[],
    'h1_avg':[],
    'h1_median':[],
    'tissue_area':[],
    'hole_area':[],
    'kc_area':[],
    'kc_nb' : [],
    'avg_kc_area':[],
    'median_kc_area':[],
    'f480_vsig4_plus_nb':[],
    'f480_vsig4_plus_median_area':[],
    'f480_vsig4_minus_nb':[],
    'f480_vsig4_minus_median_area':[],
    'h1_median_plus':[],
    'h1_median_min':[],
}

zonation_data = {
    'id':[],
    'scene':[],
    'h0_avg':[],
    'h0_median':[],
    'h1_avg':[],
    'h1_median':[],
    'h1_avg_plus':[],
    'h1_median_plus':[],
    'h1_avg_min':[],
    'h1_median_min':[],
}


### Compute zonation coeficients

In [3]:
for file in files:
    id, scene = file.split('.')[0].split('_')
    print(f'Processing {file}, id {id}, scene {scene}')
    try:
        veins = io.imread(os.path.join(vein_dir, file)).astype(bool)
        kc_f480 = io.imread(os.path.join(f480_dir, file)).astype(bool)
        kc_vsig4 = io.imread(os.path.join(vsig4_dir, file)).astype(bool)
        tissue = io.imread(os.path.join(tissue_dir, file)).astype(bool)
        glul = io.imread(os.path.join(glul_dir, file)).astype(bool)
        crap = io.imread(os.path.join(crap_dir, file)).astype(bool)
        f480_subset = io.imread(os.path.join(f480_subset_dir, file))
    except FileNotFoundError:
        print(f'File {file} not found in one of the directories, skipping')
        continue
    
    tissue = tissue & ~crap
    veins = veins & tissue
    f480_vsig4_plus = f480_subset == 255
    f480_vsig4_minus = f480_subset == 127
    f480_vsig4_plus = f480_vsig4_plus & tissue & ~veins
    f480_vsig4_minus = f480_vsig4_minus & tissue & ~veins
    kc_f480 = kc_f480 & tissue & ~veins
    kc_vsig4 = kc_vsig4 & tissue & ~veins

    glul = remove_small_objects(glul, min_size=5000)
    dist_from_glul = distance_transform_edt(~glul) * tissue * ~veins
    h0 = dist_from_glul[dist_from_glul != 0]
    h0_avg = h0.mean()
    h0_median = np.median(h0)

    kc_segment_vsig4 = np.array([r.centroid for r in regionprops(label(kc_vsig4))]).astype(int)
    kc_segment_dist_from_glul_vsig4 = dist_from_glul[kc_segment_vsig4[:,0], kc_segment_vsig4[:,1]]

    h1_avg_vsig4 = kc_segment_dist_from_glul_vsig4.mean()
    h1_median_vsig4 = np.median(kc_segment_dist_from_glul_vsig4)

    kc_segment_f480 = np.array([r.centroid for r in regionprops(label(kc_f480))]).astype(int)
    kc_segment_dist_from_glul_f480 = dist_from_glul[kc_segment_f480[:,0], kc_segment_f480[:,1]]

    h1_avg_f480 = kc_segment_dist_from_glul_f480.mean()
    h1_median_f480 = np.median(kc_segment_dist_from_glul_f480)

    h1_avg_plus = np.mean(dist_from_glul[f480_vsig4_plus])
    h1_median_plus = np.median(dist_from_glul[f480_vsig4_plus])
    h1_avg_min = np.mean(dist_from_glul[f480_vsig4_minus])
    h1_median_min = np.median(dist_from_glul[f480_vsig4_minus])


    zonation_data['id'].append(id)
    zonation_data['scene'].append(scene)
    zonation_data['h0_avg'].append(h0_avg)
    zonation_data['h0_median'].append(h0_median)
    zonation_data['h1_median_plus'].append(h1_median_plus)
    zonation_data['h1_avg_plus'].append(h1_avg_plus)
    zonation_data['h1_avg_min'].append(h1_avg_min)
    zonation_data['h1_median_min'].append(h1_median_min)
    zonation_data['h1_avg'].append(h1_avg_f480)
    zonation_data['h1_median'].append(h1_median_f480)


df = pd.DataFrame(zonation_data)
df.to_csv('kc_zonation_spatial_data.csv', index=False)

Processing 24868_scene0.png, id 24868, scene scene0
Processing 24868_scene1.png, id 24868, scene scene1
Processing 24869_scene1.png, id 24869, scene scene1
Processing 24869_scene0.png, id 24869, scene scene0
Processing 24870_scene0.png, id 24870, scene scene0
Processing 24870_scene1.png, id 24870, scene scene1
Processing 24871_scene1.png, id 24871, scene scene1
Processing 24871_scene0.png, id 24871, scene scene0
Processing 24876_scene0.png, id 24876, scene scene0
Processing 24876_scene1.png, id 24876, scene scene1
Processing 24877_scene1.png, id 24877, scene scene1
Processing 24878_scene0.png, id 24878, scene scene0
Processing 24878_scene1.png, id 24878, scene scene1
Processing 24879_scene1.png, id 24879, scene scene1
Processing 24879_scene0.png, id 24879, scene scene0
Processing 24880_scene0.png, id 24880, scene scene0
Processing 24880_scene1.png, id 24880, scene scene1
Processing 24881_scene0.png, id 24881, scene scene0
Processing 24882_scene0.png, id 24882, scene scene0
Processing 2

### Compute general properties

In [4]:
for file in files:
    id, scene = file.split('.')[0].split('_')
    print(f'Processing {file}, id {id}, scene {scene}')
    try:
        veins = io.imread(os.path.join(vein_dir, file)).astype(bool)
        kc_f480 = io.imread(os.path.join(f480_dir, file)).astype(bool)
        kc_vsig4 = io.imread(os.path.join(vsig4_dir, file)).astype(bool)
        tissue = io.imread(os.path.join(tissue_dir, file)).astype(bool)
        glul = io.imread(os.path.join(glul_dir, file)).astype(bool)
        crap = io.imread(os.path.join(crap_dir, file)).astype(bool)
        f480_subset = io.imread(os.path.join(f480_subset_dir, file))
    except FileNotFoundError:
        print(f'File {file} not found in one of the directories, skipping')
        continue
    
    tissue = tissue & ~crap
    veins = veins & tissue
    f480_vsig4_plus = f480_subset == 255
    f480_vsig4_minus = f480_subset == 127
    f480_vsig4_plus = f480_vsig4_plus & tissue & ~veins
    f480_vsig4_minus = f480_vsig4_minus & tissue & ~veins
    kc_f480 = kc_f480 & tissue & ~veins
    kc_vsig4 = kc_vsig4 & tissue & ~veins

    tissue_area = tissue.sum()
    hole_area = (veins).sum()
    kc_area_vsig4 = kc_vsig4.sum()
    kc_nb_vsig4 = label(kc_vsig4).max()
    kc_area_f480 = kc_f480.sum()
    kc_nb_f480 = label(kc_f480).max()
    f480_vsig4_plus_nb = label(f480_vsig4_plus).max()
    f480_vsig4_minus_nb = label(f480_vsig4_minus).max()

    glul = remove_small_objects(glul, min_size=5000)
    dist_from_glul = distance_transform_edt(~glul) * tissue * ~veins
    h0 = dist_from_glul[dist_from_glul != 0]
    h0_avg = h0.mean()
    h0_median = np.median(h0)

    kc_segment_vsig4 = np.array([r.centroid for r in regionprops(label(kc_vsig4))]).astype(int)
    kc_segment_dist_from_glul_vsig4 = dist_from_glul[kc_segment_vsig4[:,0], kc_segment_vsig4[:,1]]

    h1_avg_vsig4 = kc_segment_dist_from_glul_vsig4.mean()
    h1_median_vsig4 = np.median(kc_segment_dist_from_glul_vsig4)

    kc_segment_f480 = np.array([r.centroid for r in regionprops(label(kc_f480))]).astype(int)
    kc_segment_dist_from_glul_f480 = dist_from_glul[kc_segment_f480[:,0], kc_segment_f480[:,1]]

    h1_avg_f480 = kc_segment_dist_from_glul_f480.mean()
    h1_median_f480 = np.median(kc_segment_dist_from_glul_f480)

    h1_median_plus = np.median(dist_from_glul[f480_vsig4_plus])
    h1_median_min = np.median(dist_from_glul[f480_vsig4_minus])

    vsig_areas = [r.area for r in regionprops(label(kc_vsig4))]
    f480_areas = [r.area for r in regionprops(label(kc_f480))]
    avg_kc_area_vsig4 = np.mean(vsig_areas)
    median_kc_area_vsig4 = np.median(vsig_areas)
    avg_kc_area_f480 = np.mean(f480_areas)
    median_kc_area_f480 = np.median(f480_areas)

    f480_vsig4_plus_areas = [r.area for r in regionprops(label(f480_vsig4_plus))]
    f480_vsig4_minus_areas = [r.area for r in regionprops(label(f480_vsig4_minus))]
    f480_vsig4_plus_median_area = np.median(f480_vsig4_plus_areas)
    f480_vsig4_minus_median_area = np.median(f480_vsig4_minus_areas)

    data_dict_vsig4['h0_avg'].append(h0_avg)
    data_dict_vsig4['h0_median'].append(h0_median)
    data_dict_vsig4['h1_avg'].append(h1_avg_vsig4)
    data_dict_vsig4['h1_median'].append(h1_median_vsig4)
    data_dict_vsig4['id'].append(id)
    data_dict_vsig4['scene'].append(scene)
    data_dict_vsig4['tissue_area'].append(tissue_area)
    data_dict_vsig4['hole_area'].append(hole_area)
    data_dict_vsig4['kc_area'].append(kc_area_vsig4)
    data_dict_vsig4['kc_nb'].append(kc_nb_vsig4)
    data_dict_vsig4['avg_kc_area'].append(avg_kc_area_vsig4)
    data_dict_vsig4['median_kc_area'].append(median_kc_area_vsig4)

    data_dict_f480['h0_avg'].append(h0_avg)
    data_dict_f480['h0_median'].append(h0_median)
    data_dict_f480['h1_avg'].append(h1_avg_f480)
    data_dict_f480['h1_median'].append(h1_median_f480)
    data_dict_f480['id'].append(id)
    data_dict_f480['scene'].append(scene)
    data_dict_f480['tissue_area'].append(tissue_area)
    data_dict_f480['hole_area'].append(hole_area)
    data_dict_f480['kc_area'].append(kc_area_f480)
    data_dict_f480['kc_nb'].append(kc_nb_f480)
    data_dict_f480['avg_kc_area'].append(avg_kc_area_f480)
    data_dict_f480['median_kc_area'].append(median_kc_area_f480)
    data_dict_f480['f480_vsig4_plus_nb'].append(f480_vsig4_plus_nb)
    data_dict_f480['f480_vsig4_minus_nb'].append(f480_vsig4_minus_nb)
    data_dict_f480['f480_vsig4_plus_median_area'].append(f480_vsig4_plus_median_area)
    data_dict_f480['f480_vsig4_minus_median_area'].append(f480_vsig4_minus_median_area)
    data_dict_f480['h1_median_plus'].append(h1_median_plus)
    data_dict_f480['h1_median_min'].append(h1_median_min)


df_vsig4 = pd.DataFrame(data_dict_vsig4)
df_f480 = pd.DataFrame(data_dict_f480)
df_vsig4.to_csv('kc_vsig4_spatial_data.csv', index=False)
df_f480.to_csv('kc_f480_spatial_data.csv', index=False)

Processing 24868_scene0.png, id 24868, scene scene0
Processing 24868_scene1.png, id 24868, scene scene1
Processing 24869_scene1.png, id 24869, scene scene1
Processing 24869_scene0.png, id 24869, scene scene0
Processing 24870_scene0.png, id 24870, scene scene0
Processing 24870_scene1.png, id 24870, scene scene1
Processing 24871_scene1.png, id 24871, scene scene1
Processing 24871_scene0.png, id 24871, scene scene0
Processing 24876_scene0.png, id 24876, scene scene0
Processing 24876_scene1.png, id 24876, scene scene1
Processing 24877_scene1.png, id 24877, scene scene1
Processing 24878_scene0.png, id 24878, scene scene0
Processing 24878_scene1.png, id 24878, scene scene1
Processing 24879_scene1.png, id 24879, scene scene1
Processing 24879_scene0.png, id 24879, scene scene0
Processing 24880_scene0.png, id 24880, scene scene0
Processing 24880_scene1.png, id 24880, scene scene1
Processing 24881_scene0.png, id 24881, scene scene0
Processing 24882_scene0.png, id 24882, scene scene0
Processing 2